# 🥈 Bronze → Silver

Transforma Bronze em Silver: limpeza, tipagem, features e dimensões básicas.

In [ ]:
import os
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'SBD2', cwd.parent]
    for root in candidates:
        if (root / 'data').exists() and (root / 'notebooks').exists():
            return root
        if (root / 'Crime_Data_from_2020_to_Present.csv').exists():
            return root
    return cwd

PROJECT_ROOT = find_project_root()
BRONZE_PATH = PROJECT_ROOT / 'data' / 'bronze' / 'crime_data_bronze.parquet'
SILVER_DIR = PROJECT_ROOT / 'data' / 'silver'
SILVER_DIR.mkdir(parents=True, exist_ok=True)

BATCH_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

print('📁 Projeto:', PROJECT_ROOT)
print('📄 Bronze:', BRONZE_PATH)
print('📁 Silver:', SILVER_DIR)
print('🔖 Batch:', BATCH_ID)

if not BRONZE_PATH.exists():
    raise FileNotFoundError(f'Arquivo Bronze não encontrado: {BRONZE_PATH}')

In [ ]:
df_bronze = pd.read_parquet(BRONZE_PATH)
print('✅ Bronze carregado', df_bronze.shape)

metadata_cols = [c for c in df_bronze.columns if str(c).startswith('_')]
data_cols = [c for c in df_bronze.columns if c not in metadata_cols]
df = df_bronze[data_cols].copy()
print('📊 dados:', len(data_cols), '| metadados:', len(metadata_cols))
df.head()

In [ ]:
# Tipagens principais
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], errors='coerce')
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')

df['TIME OCC'] = df['TIME OCC'].astype(str).str.zfill(4)
df['HOUR'] = pd.to_numeric(df['TIME OCC'].str[:2], errors='coerce')

numeric_columns = [
    'DR_NO','AREA','Rpt Dist No','Part 1-2','Crm Cd','Vict Age','Premis Cd',
    'Weapon Used Cd','Crm Cd 1','Crm Cd 2','Crm Cd 3','Crm Cd 4','LAT','LON'
]
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('✅ Tipos convertidos')
df.dtypes

In [ ]:
# Limpezas básicas
critical = ['DR_NO', 'DATE OCC', 'Crm Cd', 'AREA', 'AREA NAME']
df = df.dropna(subset=[c for c in critical if c in df.columns])

# coords inválidas
invalid_coords = (df['LAT'] == 0) | (df['LON'] == 0) | (df['LAT'] < 33.5) | (df['LAT'] > 35) | (df['LON'] < -119) | (df['LON'] > -117)
df.loc[invalid_coords, ['LAT','LON']] = np.nan

# idade inválida
invalid_age = (df['Vict Age'] < 0) | (df['Vict Age'] > 120)
df.loc[invalid_age, 'Vict Age'] = np.nan

fill_values = {
    'Vict Sex': 'U',
    'Vict Descent': 'X',
    'Weapon Desc': 'NONE',
    'Premis Desc': 'UNKNOWN',
    'Status Desc': 'UNKNOWN',
    'Cross Street': 'UNKNOWN',
}
for col, v in fill_values.items():
    if col in df.columns:
        df[col] = df[col].fillna(v)

print('✅ Limpeza básica aplicada:', df.shape)

In [ ]:
# Features
df['YEAR'] = df['DATE OCC'].dt.year
df['MONTH'] = df['DATE OCC'].dt.month
df['DAY'] = df['DATE OCC'].dt.day
df['DAY_OF_WEEK'] = df['DATE OCC'].dt.dayofweek
df['IS_WEEKEND'] = df['DAY_OF_WEEK'].isin([5,6]).astype(int)

def period_of_day(h):
    if pd.isna(h):
        return 'UNKNOWN'
    h = int(h)
    if 0 <= h < 6: return 'MADRUGADA'
    if 6 <= h < 12: return 'MANHA'
    if 12 <= h < 18: return 'TARDE'
    return 'NOITE'

df['PERIOD'] = df['HOUR'].apply(period_of_day)

violent_codes = {110,113,121,122,210,220,230,231,235,236,250,251,310,320,510,520}
df['IS_VIOLENT'] = df['Crm Cd'].isin(violent_codes).astype(int)

print('✅ Features criadas')
df[['DATE OCC','HOUR','PERIOD','IS_VIOLENT']].head()

In [ ]:
# Normalização de nomes
column_mapping = {
  'DR_NO':'crime_id',
  'Date Rptd':'date_reported',
  'DATE OCC':'date_occurred',
  'TIME OCC':'time_occurred',
  'HOUR':'hour_occurred',
  'AREA':'area_code',
  'AREA NAME':'area_name',
  'Rpt Dist No':'district_number',
  'Part 1-2':'part_code',
  'Crm Cd':'crime_code',
  'Crm Cd Desc':'crime_description',
  'Vict Age':'victim_age',
  'Vict Sex':'victim_sex',
  'Vict Descent':'victim_descent',
  'Premis Cd':'premise_code',
  'Premis Desc':'premise_description',
  'Weapon Used Cd':'weapon_code',
  'Weapon Desc':'weapon_description',
  'Status':'status_code',
  'Status Desc':'status_description',
  'LOCATION':'location',
  'Cross Street':'cross_street',
  'LAT':'latitude',
  'LON':'longitude',
  'YEAR':'year_occurred',
  'MONTH':'month_occurred',
  'DAY':'day_occurred',
  'DAY_OF_WEEK':'day_of_week',
  'IS_WEEKEND':'is_weekend',
  'PERIOD':'period_of_day',
  'IS_VIOLENT':'is_violent'
}
df = df.rename(columns=column_mapping)

df['_silver_timestamp'] = datetime.now()
df['_silver_batch_id'] = BATCH_ID

print('✅ Colunas normalizadas', df.shape)
df.head()

In [ ]:
# Dimensões Silver (básicas)
dim_areas = df[['area_code','area_name']].drop_duplicates().sort_values('area_code').reset_index(drop=True)
dim_crime_types = df[['crime_code','crime_description','is_violent']].drop_duplicates().sort_values('crime_code').reset_index(drop=True)
dim_weapons = df[['weapon_code','weapon_description']].drop_duplicates().dropna(subset=['weapon_code']).sort_values('weapon_code').reset_index(drop=True)
dim_premises = df[['premise_code','premise_description']].drop_duplicates().dropna(subset=['premise_code']).sort_values('premise_code').reset_index(drop=True)

print('✅ dims:', len(dim_areas), len(dim_crime_types), len(dim_weapons), len(dim_premises))

In [ ]:
# Persistência Silver
crimes_path = SILVER_DIR / 'crimes.parquet'
used_compression = 'snappy'
try:
    df.to_parquet(crimes_path, index=False, compression=used_compression)
except Exception as e:
    print(f"⚠️ Falha com {used_compression}. Tentando gzip. Erro: {e}")
    used_compression = 'gzip'
    df.to_parquet(crimes_path, index=False, compression=used_compression)

dim_areas.to_parquet(SILVER_DIR / 'dim_areas.parquet', index=False)
dim_crime_types.to_parquet(SILVER_DIR / 'dim_crime_types.parquet', index=False)
dim_weapons.to_parquet(SILVER_DIR / 'dim_weapons.parquet', index=False)
dim_premises.to_parquet(SILVER_DIR / 'dim_premises.parquet', index=False)

df.to_csv(SILVER_DIR / 'crimes.csv', index=False)

meta = {
  'batch_id': BATCH_ID,
  'transformation_timestamp': datetime.now().isoformat(),
  'source_layer': 'bronze',
  'target_layer': 'silver',
  'total_records': int(len(df)),
  'compression': used_compression,
}
meta_path = SILVER_DIR / f'transformation_metadata_{BATCH_ID}.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, default=str)

print('✅ Silver salvo:', crimes_path)
print('🔍 Verificação leitura:', 'OK' if len(pd.read_parquet(crimes_path)) == len(df) else 'FALHA')

# 🥈 Bronze → Silver
## Crime Data from 2020 to Present - Los Angeles

### Arquitetura Medalhão - Camada Silver

Este notebook realiza a transformação dos dados da camada Bronze para a camada Silver.

**Camada Silver**: Dados limpos, validados, normalizados e enriquecidos com features derivadas.

### Transformações:
1. Limpeza de dados (tipos, valores nulos)
2. Validação e correção de inconsistências
3. Normalização de campos
4. Feature Engineering
5. Criação de tabelas dimensionais
6. Persistência na camada Silver

In [ ]:
# Importações
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# Importar módulos locais (robusto a diferentes CWDs)
import sys
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'SBD2', cwd.parent]
    for root in candidates:
        if (root / 'src').exists() and (root / 'notebooks').exists():
            return root
    return cwd

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_processing import *  # noqa: F401,F403

# Configurações
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('✅ Bibliotecas carregadas com sucesso!')
print(f'📅 Data de execução: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'📁 Projeto: {PROJECT_ROOT}')

## 1. Configuração de Caminhos

In [ ]:
# Configuração de diretórios
BRONZE_DATA_PATH = PROJECT_ROOT / 'data' / 'bronze' / 'crime_data_bronze.parquet'
SILVER_DATA_DIR = PROJECT_ROOT / 'data' / 'silver'

# Criar diretório Silver se não existir
SILVER_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Parâmetros de transformação
BATCH_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f'📁 Bronze: {BRONZE_DATA_PATH}')
print(f'📁 Silver: {SILVER_DATA_DIR}')
print(f'🔖 Batch ID: {BATCH_ID}')

## 2. Carregamento dos Dados Bronze

In [ ]:
# Carregar dados da camada Bronze
if not BRONZE_DATA_PATH.exists():
    raise FileNotFoundError(f"Arquivo Bronze não encontrado: {BRONZE_DATA_PATH}")

try:
    df_bronze = pd.read_parquet(BRONZE_DATA_PATH)
except Exception as e:
    raise RuntimeError(
        "Falha ao ler Parquet da camada Bronze. "
        "Instale 'pyarrow' (recomendado) ou 'fastparquet'. "
        f"Erro: {e}"
    )

print(f"✅ Dados Bronze carregados!")
print(f"📊 Shape: {df_bronze.shape[0]:,} linhas x {df_bronze.shape[1]} colunas")
print(f"💾 Memória: {df_bronze.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Verificar metadados de ingestão
print("📋 Metadados de Ingestão:")
if '_batch_id' in df_bronze.columns:
    print(f"   🔖 Batch ID: {df_bronze['_batch_id'].iloc[0]}")
if '_ingestion_timestamp' in df_bronze.columns:
    print(f"   🕐 Ingestão: {df_bronze['_ingestion_timestamp'].iloc[0]}")
if '_source_system' in df_bronze.columns:
    print(f"   📂 Fonte: {df_bronze['_source_system'].iloc[0]}")

In [ ]:
# Separar dados dos metadados
metadata_cols = [col for col in df_bronze.columns if col.startswith('_')]
data_cols = [col for col in df_bronze.columns if not col.startswith('_')]

# Manter apenas as colunas de dados para processamento
df = df_bronze[data_cols].copy()

print(f"📊 Colunas de dados: {len(data_cols)}")
print(f"📋 Colunas de metadados: {len(metadata_cols)}")

## 3. Limpeza de Tipos de Dados

In [ ]:
# Verificar tipos atuais
print("📊 Tipos de dados atuais:")
df.dtypes

In [ ]:
# Converter colunas de data
print("🔄 Convertendo colunas de data...")

# Date Reported
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')

# Date Occurred
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')

# Time Occurred - converter para hora
df['TIME OCC'] = df['TIME OCC'].astype(str).str.zfill(4)
df['HOUR'] = df['TIME OCC'].str[:2].astype(int)

print("✅ Colunas de data convertidas!")
print(f"   📅 Date Rptd - Valores válidos: {df['Date Rptd'].notna().sum():,}")
print(f"   📅 DATE OCC - Valores válidos: {df['DATE OCC'].notna().sum():,}")

In [ ]:
# Converter colunas numéricas
print("🔄 Convertendo colunas numéricas...")

numeric_columns = ['DR_NO', 'AREA', 'Rpt Dist No', 'Crm Cd', 'Vict Age', 
                   'Premis Cd', 'Weapon Used Cd', 'Crm Cd 1', 'Crm Cd 2', 
                   'Crm Cd 3', 'Crm Cd 4']

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Coordenadas geográficas
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')

print("✅ Colunas numéricas convertidas!")

## 4. Tratamento de Valores Ausentes

In [ ]:
# Verificar valores nulos antes
null_before = df.isnull().sum()
print("📊 Valores nulos antes do tratamento:")
null_before[null_before > 0].sort_values(ascending=False)

In [ ]:
# Definir colunas críticas (obrigatórias)
CRITICAL_COLUMNS = ['DR_NO', 'DATE OCC', 'Crm Cd', 'AREA', 'AREA NAME']

print(f"📊 Registros antes: {len(df):,}")

# Remover registros com valores nulos em colunas críticas
df = df.dropna(subset=[c for c in CRITICAL_COLUMNS if c in df.columns])

print(f"📊 Registros após remoção de nulos críticos: {len(df):,}")

In [ ]:
# Tratar coordenadas inválidas (0, 0)
print("🗺️ Tratando coordenadas inválidas...")

# Coordenadas de Los Angeles: aproximadamente LAT 33.7-34.8, LON -118.7 a -117.9
invalid_coords = (df['LAT'] == 0) | (df['LON'] == 0) | \
                 (df['LAT'] < 33.5) | (df['LAT'] > 35) | \
                 (df['LON'] < -119) | (df['LON'] > -117)

print(f"   ⚠️ Coordenadas inválidas: {invalid_coords.sum():,}")

# Substituir por NaN (não remover, apenas marcar como ausente)
df.loc[invalid_coords, 'LAT'] = np.nan
df.loc[invalid_coords, 'LON'] = np.nan

print(f"   ✅ Coordenadas inválidas tratadas")

In [ ]:
# Tratar idade das vítimas
print("👤 Tratando idade das vítimas...")

# Idades inválidas (negativas ou > 120)
invalid_ages = (df['Vict Age'] < 0) | (df['Vict Age'] > 120)
print(f"   ⚠️ Idades inválidas: {invalid_ages.sum():,}")

# Substituir idades inválidas por NaN
df.loc[invalid_ages, 'Vict Age'] = np.nan

print(f"   ✅ Idades inválidas tratadas")

In [ ]:
# Preencher valores ausentes em campos descritivos
fill_values = {
    'Vict Sex': 'X',           # X = Unknown
    'Vict Descent': 'X',       # X = Unknown
    'Weapon Desc': 'NONE',
    'Cross Street': 'UNKNOWN',
    'Premis Desc': 'UNKNOWN',
    'Status Desc': 'UNKNOWN'
}

for col, value in fill_values.items():
    if col in df.columns:
        df[col] = df[col].fillna(value)

print("✅ Valores ausentes preenchidos em campos descritivos")

## 5. Feature Engineering

In [ ]:
# Criar features temporais
print("🕐 Criando features temporais...")

df['YEAR'] = df['DATE OCC'].dt.year
df['MONTH'] = df['DATE OCC'].dt.month
df['DAY'] = df['DATE OCC'].dt.day
df['DAY_OF_WEEK'] = df['DATE OCC'].dt.dayofweek
df['DAY_NAME'] = df['DATE OCC'].dt.day_name()
df['WEEK_OF_YEAR'] = df['DATE OCC'].dt.isocalendar().week
df['QUARTER'] = df['DATE OCC'].dt.quarter
df['IS_WEEKEND'] = df['DAY_OF_WEEK'].isin([5, 6]).astype(int)

print("✅ Features temporais criadas!")

In [ ]:
# Criar período do dia
print("🌙 Criando período do dia...")

def get_period(hour):
    if pd.isna(hour):
        return 'UNKNOWN'
    hour = int(hour)
    if 0 <= hour < 6:
        return 'MADRUGADA'
    elif 6 <= hour < 12:
        return 'MANHA'
    elif 12 <= hour < 18:
        return 'TARDE'
    else:
        return 'NOITE'

df['PERIOD'] = df['HOUR'].apply(get_period)

print("✅ Período do dia criado!")
print(df['PERIOD'].value_counts())

In [ ]:
# Criar faixa etária
print("👥 Criando faixas etárias...")

def get_age_group(age):
    if pd.isna(age):
        return 'UNKNOWN'
    age = int(age)
    if age < 18:
        return 'MENOR'
    elif age < 30:
        return 'JOVEM'
    elif age < 45:
        return 'ADULTO'
    elif age < 60:
        return 'MEIA_IDADE'
    else:
        return 'IDOSO'

df['AGE_GROUP'] = df['Vict Age'].apply(get_age_group)

print("✅ Faixas etárias criadas!")
print(df['AGE_GROUP'].value_counts())

In [ ]:
# Classificar crimes violentos
print("⚠️ Classificando crimes violentos...")

# Códigos de crimes violentos (homicídio, assalto, roubo, etc.)
VIOLENT_CRIME_CODES = [
    110,  # CRIMINAL HOMICIDE
    113,  # MANSLAUGHTER, NEGLIGENT
    121,  # RAPE, FORCIBLE
    122,  # RAPE, ATTEMPTED
    210,  # ROBBERY
    220,  # ATTEMPTED ROBBERY
    230,  # ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT
    231,  # ASSAULT WITH DEADLY WEAPON ON POLICE OFFICER
    235,  # CHILD ABUSE (PHYSICAL)
    236,  # INTIMATE PARTNER - SIMPLE ASSAULT
    250,  # SHOTS FIRED AT INHABITED DWELLING
    251,  # SHOTS FIRED AT MOVING VEHICLE, TRAIN OR AIRCRAFT
    310,  # BURGLARY
    320,  # BURGLARY FROM VEHICLE
    510,  # VEHICLE - STOLEN
    520,  # VEHICLE - ATTEMPT STOLEN
]

df['IS_VIOLENT'] = df['Crm Cd'].isin(VIOLENT_CRIME_CODES).astype(int)

print("✅ Crimes classificados!")
print(f"   🔴 Crimes violentos: {df['IS_VIOLENT'].sum():,} ({df['IS_VIOLENT'].mean()*100:.1f}%)")
print(f"   🟢 Crimes não-violentos: {(1-df['IS_VIOLENT']).sum():,}")

In [ ]:
# Criar categoria de arma
print("🔫 Categorizando armas...")

def categorize_weapon(desc):
    if pd.isna(desc) or desc in ['NONE', 'UNKNOWN']:
        return 'SEM_ARMA'
    desc = str(desc).upper()
    if 'GUN' in desc or 'PISTOL' in desc or 'RIFLE' in desc or 'FIREARM' in desc:
        return 'ARMA_DE_FOGO'
    elif 'KNIFE' in desc or 'CUTTING' in desc or 'MACHETE' in desc:
        return 'ARMA_BRANCA'
    elif 'VEHICLE' in desc:
        return 'VEICULO'
    elif 'HAND' in desc or 'FIST' in desc or 'FEET' in desc:
        return 'FORCA_CORPORAL'
    else:
        return 'OUTROS'

df['WEAPON_CATEGORY'] = df['Weapon Desc'].apply(categorize_weapon)

print("✅ Armas categorizadas!")
print(df['WEAPON_CATEGORY'].value_counts())

## 6. Normalização de Campos

In [ ]:
# Padronizar nomes de colunas
print("📋 Padronizando nomes de colunas...")

column_mapping = {
    'DR_NO': 'crime_id',
    'Date Rptd': 'date_reported',
    'DATE OCC': 'date_occurred',
    'TIME OCC': 'time_occurred',
    'HOUR': 'hour_occurred',
    'AREA': 'area_code',
    'AREA NAME': 'area_name',
    'Rpt Dist No': 'district_number',
    'Part 1-2': 'part_code',
    'Crm Cd': 'crime_code',
    'Crm Cd Desc': 'crime_description',
    'Mocodes': 'modus_operandi',
    'Vict Age': 'victim_age',
    'Vict Sex': 'victim_sex',
    'Vict Descent': 'victim_descent',
    'Premis Cd': 'premise_code',
    'Premis Desc': 'premise_description',
    'Weapon Used Cd': 'weapon_code',
    'Weapon Desc': 'weapon_description',
    'Status': 'status_code',
    'Status Desc': 'status_description',
    'Crm Cd 1': 'crime_code_1',
    'Crm Cd 2': 'crime_code_2',
    'Crm Cd 3': 'crime_code_3',
    'Crm Cd 4': 'crime_code_4',
    'LOCATION': 'location',
    'Cross Street': 'cross_street',
    'LAT': 'latitude',
    'LON': 'longitude',
    'YEAR': 'year_occurred',
    'MONTH': 'month_occurred',
    'DAY': 'day_occurred',
    'DAY_OF_WEEK': 'day_of_week',
    'DAY_NAME': 'day_name',
    'WEEK_OF_YEAR': 'week_of_year',
    'QUARTER': 'quarter',
    'IS_WEEKEND': 'is_weekend',
    'PERIOD': 'period_of_day',
    'AGE_GROUP': 'age_group',
    'IS_VIOLENT': 'is_violent',
    'WEAPON_CATEGORY': 'weapon_category'
}

df = df.rename(columns=column_mapping)

print(f"✅ {len(column_mapping)} colunas renomeadas!")

In [ ]:
# Padronizar valores de sexo da vítima
sex_mapping = {
    'M': 'M',  # Male
    'F': 'F',  # Female
    'X': 'U',  # Unknown
    'H': 'U',  # Unknown/Other
    '-': 'U',
    '': 'U'
}

df['victim_sex'] = df['victim_sex'].map(sex_mapping).fillna('U')

print("✅ Valores de sexo padronizados:")
print(df['victim_sex'].value_counts())

In [ ]:
# Adicionar descrição de descendência
descent_mapping = {
    'A': 'Asian',
    'B': 'Black',
    'C': 'Chinese',
    'D': 'Cambodian',
    'F': 'Filipino',
    'G': 'Guamanian',
    'H': 'Hispanic/Latin/Mexican',
    'I': 'American Indian/Alaskan Native',
    'J': 'Japanese',
    'K': 'Korean',
    'L': 'Laotian',
    'O': 'Other',
    'P': 'Pacific Islander',
    'S': 'Samoan',
    'U': 'Hawaiian',
    'V': 'Vietnamese',
    'W': 'White',
    'X': 'Unknown',
    'Z': 'Asian Indian'
}

df['victim_descent_desc'] = df['victim_descent'].map(descent_mapping).fillna('Unknown')

print("✅ Descrição de descendência adicionada!")

## 7. Criação de Tabelas Dimensionais

In [ ]:
# Dimensão: Áreas
print("🗺️ Criando dimensão de áreas...")

dim_areas = df[['area_code', 'area_name']].drop_duplicates().reset_index(drop=True)
dim_areas = dim_areas.sort_values('area_code')

# Contar crimes por área
crimes_by_area = df.groupby('area_code').size().reset_index(name='total_crimes')
dim_areas = dim_areas.merge(crimes_by_area, on='area_code', how='left')

print(f"✅ Dimensão de áreas: {len(dim_areas)} registros")
dim_areas.head(10)

In [ ]:
# Dimensão: Tipos de Crime
print("🔍 Criando dimensão de tipos de crime...")

dim_crime_types = df[['crime_code', 'crime_description', 'is_violent']].drop_duplicates().reset_index(drop=True)
dim_crime_types = dim_crime_types.sort_values('crime_code')

# Contar ocorrências
crime_counts = df.groupby('crime_code').size().reset_index(name='total_occurrences')
dim_crime_types = dim_crime_types.merge(crime_counts, on='crime_code', how='left')

print(f"✅ Dimensão de tipos de crime: {len(dim_crime_types)} registros")
dim_crime_types.head(10)

In [ ]:
# Dimensão: Armas
print("🔫 Criando dimensão de armas...")

dim_weapons = df[['weapon_code', 'weapon_description', 'weapon_category']].drop_duplicates().reset_index(drop=True)
dim_weapons = dim_weapons.dropna(subset=['weapon_code'])
dim_weapons = dim_weapons.sort_values('weapon_code')

print(f"✅ Dimensão de armas: {len(dim_weapons)} registros")
dim_weapons.head(10)

In [ ]:
# Dimensão: Locais (Premises)
print("🏢 Criando dimensão de locais...")

dim_premises = df[['premise_code', 'premise_description']].drop_duplicates().reset_index(drop=True)
dim_premises = dim_premises.dropna(subset=['premise_code'])
dim_premises = dim_premises.sort_values('premise_code')

print(f"✅ Dimensão de locais: {len(dim_premises)} registros")
dim_premises.head(10)

## 8. Adicionar Metadados Silver

In [ ]:
# Adicionar metadados de transformação
df['_silver_timestamp'] = datetime.now()
df['_silver_batch_id'] = BATCH_ID

print("✅ Metadados Silver adicionados!")

In [ ]:
# Verificar estrutura final
print(f"\n📊 Estrutura Final da Camada Silver:")
print(f"   Shape: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"\n📋 Colunas:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col}")

## 9. Persistência na Camada Silver

In [ ]:
# Salvar tabela principal de crimes
crimes_path = SILVER_DATA_DIR / 'crimes.parquet'
used_compression = 'snappy'
try:
    df.to_parquet(crimes_path, index=False, compression=used_compression)
except Exception as e:
    print(f"⚠️ Falha ao salvar com '{used_compression}'. Tentando 'gzip'. Erro: {e}")
    used_compression = 'gzip'
    df.to_parquet(crimes_path, index=False, compression=used_compression)

print(f"✅ Tabela de crimes salva!")
print(f"   📁 Caminho: {crimes_path}")
print(f"   📦 Compressão: {used_compression}")
print(f"   📊 Registros: {len(df):,}")
print(f"   💾 Tamanho: {crimes_path.stat().st_size / 1024**2:.2f} MB")

In [ ]:
# Salvar tabelas dimensionais
(SILVER_DATA_DIR / 'dim_areas.parquet').unlink(missing_ok=True)
(SILVER_DATA_DIR / 'dim_crime_types.parquet').unlink(missing_ok=True)
(SILVER_DATA_DIR / 'dim_weapons.parquet').unlink(missing_ok=True)
(SILVER_DATA_DIR / 'dim_premises.parquet').unlink(missing_ok=True)

dim_areas.to_parquet(SILVER_DATA_DIR / 'dim_areas.parquet', index=False)
dim_crime_types.to_parquet(SILVER_DATA_DIR / 'dim_crime_types.parquet', index=False)
dim_weapons.to_parquet(SILVER_DATA_DIR / 'dim_weapons.parquet', index=False)
dim_premises.to_parquet(SILVER_DATA_DIR / 'dim_premises.parquet', index=False)

print("✅ Tabelas dimensionais salvas!")
print(f"   📁 {SILVER_DATA_DIR / 'dim_areas.parquet'} ({len(dim_areas)} registros)")
print(f"   📁 {SILVER_DATA_DIR / 'dim_crime_types.parquet'} ({len(dim_crime_types)} registros)")
print(f"   📁 {SILVER_DATA_DIR / 'dim_weapons.parquet'} ({len(dim_weapons)} registros)")
print(f"   📁 {SILVER_DATA_DIR / 'dim_premises.parquet'} ({len(dim_premises)} registros)")

In [ ]:
# Salvar também em CSV para compatibilidade
df.to_csv(SILVER_DATA_DIR / 'crimes.csv', index=False)

print(f"✅ Backup CSV salvo: {SILVER_DATA_DIR / 'crimes.csv'}")

In [ ]:
# Criar arquivo de metadados da transformação
import json

silver_metadata = {
    'batch_id': BATCH_ID,
    'transformation_timestamp': datetime.now().isoformat(),
    'source_layer': 'bronze',
    'target_layer': 'silver',
    'total_records': len(df),
    'total_columns': len(df.columns),
    'columns': list(df.columns),
    'dimensions': {
        'dim_areas': len(dim_areas),
        'dim_crime_types': len(dim_crime_types),
        'dim_weapons': len(dim_weapons),
        'dim_premises': len(dim_premises),
    },
    'output_format': 'parquet',
    'compression': used_compression,
    'transformations_applied': [
        'date_conversion',
        'null_handling',
        'coordinate_validation',
        'age_validation',
        'feature_engineering',
        'column_standardization',
        'dimension_extraction',
    ],
}

metadata_path = SILVER_DATA_DIR / f'transformation_metadata_{BATCH_ID}.json'
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(silver_metadata, f, indent=2)

print(f"✅ Metadados de transformação salvos: {metadata_path}")

## 10. Verificação de Qualidade

In [ ]:
# Verificar integridade dos arquivos salvos
try:
    df_verify = pd.read_parquet(crimes_path)
    print("🔍 Verificação de Integridade:")
    print(f"   📊 Registros originais: {len(df):,}")
    print(f"   📊 Registros verificados: {len(df_verify):,}")
    print(f"   ✅ Integridade: {'OK' if len(df) == len(df_verify) else 'FALHA'}")
except Exception as e:
    raise RuntimeError(f"Falha ao ler crimes.parquet salvo na Silver: {e}")

In [ ]:
# Resumo estatístico
print("=" * 60)
print("📊 RESUMO DA TRANSFORMAÇÃO - CAMADA SILVER")
print("=" * 60)
print(f"\n🔖 Batch ID: {BATCH_ID}")
print(f"📅 Data/Hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n📊 Estatísticas:")
print(f"   • Total de crimes: {len(df):,}")
print(f"   • Período: {df['date_occurred'].min()} a {df['date_occurred'].max()}")
print(f"   • Áreas distintas: {df['area_name'].nunique()}")
print(f"   • Tipos de crime: {df['crime_code'].nunique()}")
print(f"   • Crimes violentos: {df['is_violent'].sum():,} ({df['is_violent'].mean()*100:.1f}%)")
print(f"\n📁 Arquivos gerados:")
print(f"   • {crimes_path}")
print(f"   • {SILVER_DATA_DIR}/dim_*.parquet")
print(f"   • {metadata_path}")
print("\n✅ Transformação Bronze → Silver concluída com sucesso!")

## 📋 Resumo - Bronze → Silver

### Transformações realizadas:
- ✅ Conversão de tipos de dados (datas, numéricos)
- ✅ Tratamento de valores nulos
- ✅ Validação de coordenadas geográficas
- ✅ Validação de idades
- ✅ Feature Engineering:
  - Features temporais (ano, mês, dia, trimestre, etc.)
  - Período do dia
  - Faixa etária
  - Classificação de crime violento
  - Categoria de arma
- ✅ Normalização de nomes de colunas
- ✅ Padronização de valores categóricos
- ✅ Criação de tabelas dimensionais
- ✅ Persistência em formato Parquet

### Próximo passo:
Execute o notebook `03_silver_to_gold.ipynb` para criar o modelo dimensional na camada Gold.